# HW4 — Applied: Tracking by Detection + Optical Flow

**Due:** July 22, 2026

A good detector finds every person in a frame — but it is **expensive**, far too slow to run on every frame
of a video in real time. Optical flow is cheap. So real trackers split the work: the **detector** finds people
(and, run again later, discovers anyone who has since walked in), while **optical flow** cheaply carries each
box from frame to frame in between. How often you re-run the detector — every `K` frames — trades processing
speed against how quickly a newcomer gets noticed.

You'll use a pretrained **Faster R-CNN** detector and the **pyramidal Lucas–Kanade** optical-flow tracker
(provided below, so this notebook stands alone). Your job is the step that joins them — carrying a box with the
flow — and then to measure the speed/quality tradeoff in `K`, including the price of detecting every frame.

Everything is **PyTorch** (no NumPy). Submit the notebook **with its video outputs intact**.

In [ ]:
STUDENT_ID = ""   # <-- fill in

## Provided tools — read them, don't reimplement

Video I/O, the pretrained detector, the Lucas–Kanade tracker, and small helpers to seed points and to show
video/images inline. All take and return plain `torch` tensors.

> **Setup:** install the dependencies once with `pip install -r requirements.txt`. The setup cell below picks a GPU automatically when one is available, downloads the detector weights on first run (needs internet), and fetches the sample clip if it isn't already present.

In [ ]:
import torch, torch.nn.functional as F, av, tempfile
from torchvision.utils import draw_bounding_boxes, save_image
from torchvision.models.detection import (fasterrcnn_mobilenet_v3_large_fpn,
                                           FasterRCNN_MobileNet_V3_Large_FPN_Weights)
from IPython.display import Video, Image as IImage

def load_clip(path, max_frames=120):                 # -> frames [T,H,W,3] uint8, fps
    c = av.open(path); fr = []
    for f in c.decode(video=0):
        fr.append(torch.from_numpy(f.to_ndarray(format="rgb24")))
        if len(fr) >= max_frames: break
    fps = float(c.streams.video[0].average_rate); c.close()
    return torch.stack(fr), fps

def write_clip(frames, path, fps):                   # frames [T,H,W,3] uint8 -> mp4
    c = av.open(path, "w"); s = c.add_stream("libx264", rate=int(round(fps)))
    s.width, s.height, s.pix_fmt = frames.shape[2], frames.shape[1], "yuv420p"
    for f in frames:
        for p in s.encode(av.VideoFrame.from_ndarray(f.numpy(), format="rgb24")): c.mux(p)
    for p in s.encode(): c.mux(p)
    c.close()

def draw(frame, boxes, color="lime"):                # frame [H,W,3] uint8 -> [H,W,3] uint8
    if len(boxes) == 0: return frame
    return draw_bounding_boxes(frame.permute(2,0,1), boxes, colors=color, width=3).permute(1,2,0)

def show(frames, fps=10, width=520):                 # inline video
    p = tempfile.mktemp(suffix=".mp4"); write_clip(frames, p, fps)
    return Video(p, embed=True, width=width, html_attributes="controls loop autoplay muted")

def show_frame(frame, width=520):                    # inline single image
    p = tempfile.mktemp(suffix=".png"); save_image(frame.permute(2,0,1).float()/255, p)
    return IImage(p, width=width)

In [ ]:
import os, urllib.request

# --- device: CPU now, GPU automatically when one is present (e.g. the grading VM) ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("running on", DEVICE)

# --- data dir: a "data" folder next to this notebook (Jupyter runs the notebook from its own folder) ---
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
VTEST = os.path.join(DATA_DIR, "vtest.avi")

# --- fetch the sample clip once if it isn't already here (pinned to a fixed OpenCV tag) ---
_VTEST_URL = "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/vtest.avi"
if not os.path.exists(VTEST):
    print("downloading vtest.avi ...")
    urllib.request.urlretrieve(_VTEST_URL, VTEST)
print("clip at", VTEST)

In [ ]:
_w = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
detector = fasterrcnn_mobilenet_v3_large_fpn(weights=_w).eval().to(DEVICE)

@torch.no_grad()
def detect_people(frame, score=0.7):                 # frame [H,W,3] uint8 -> boxes [N,4]
    out = detector([frame.permute(2,0,1).float().to(DEVICE)/255])[0]
    keep = (out["labels"] == 1) & (out["scores"] > score)   # COCO label 1 = person
    return out["boxes"][keep].cpu()

In [ ]:
def to_gray(frames): return frames.float().mean(-1) / 255           # [T,H,W,3] -> [T,H,W]

def _grad(I):
    k = torch.tensor([[-1.,0,1],[-2,0,2],[-1,0,1]]).view(1,1,3,3) / 8
    return F.conv2d(I, k, padding=1), F.conv2d(I, k.transpose(-1,-2), padding=1)

def _sample(I, p):
    H, W = I.shape[-2:]
    g = torch.stack([p[...,0]/(W-1)*2-1, p[...,1]/(H-1)*2-1], -1)
    return F.grid_sample(I, g.view(1,-1,1,2), align_corners=True, padding_mode="border").view(p.shape[:-1])

def lk_track(g0, g1, pts, win=15, iters=10, levels=4, eig=1e-3):    # pyramidal inverse-compositional LK
    r = win//2; a = torch.arange(-r, r+1).float()
    oy, ox = torch.meshgrid(a, a, indexing="ij"); off = torch.stack([ox.reshape(-1), oy.reshape(-1)], -1)
    P0, P1 = [g0], [g1]
    for _ in range(levels-1):
        P0.append(F.avg_pool2d(P0[-1][None,None], 2)[0,0]); P1.append(F.avg_pool2d(P1[-1][None,None], 2)[0,0])
    d = torch.zeros_like(pts)
    for l in reversed(range(levels)):
        I0, I1 = P0[l][None,None], P1[l][None,None]; Ix, Iy = _grad(I0)
        anchor = pts / 2**l; ap = anchor[:,None,:] + off                # template window pinned at the seed
        T0 = _sample(I0, ap); gx = _sample(Ix, ap); gy = _sample(Iy, ap)  # template + gradients: computed once
        Gxx, Gyy, Gxy = (gx*gx).sum(1), (gy*gy).sum(1), (gx*gy).sum(1)
        det = (Gxx*Gyy - Gxy*Gxy).clamp_min(1e-4)
        for _ in range(iters):
            wp = (anchor + d)[:,None,:] + off                           # only the I1 window moves
            t = _sample(I1, wp) - T0
            bx, by = (gx*t).sum(1), (gy*t).sum(1)
            d = d + torch.stack([-(Gyy*bx - Gxy*by)/det, -(-Gxy*bx + Gxx*by)/det], -1)
        if l > 0: d = d * 2
    p = pts + d
    mineig = (Gxx+Gyy)/2 - ((Gxx-Gyy)**2/4 + Gxy**2).clamp_min(0).sqrt()   # drop low-texture (untrackable) points
    valid = (mineig > eig) & (p[:,0] >= 1) & (p[:,0] < g0.shape[1]-1) & (p[:,1] >= 1) & (p[:,1] < g0.shape[0]-1)
    return p, valid

def seed_points(box, n=8, m=0.12):                        # grid of n*n points inside a box
    x1, y1, x2, y2 = box; w, h = x2-x1, y2-y1
    xs = torch.linspace(x1+m*w, x2-m*w, n); ys = torch.linspace(y1+m*h, y2-m*h, n)
    gx, gy = torch.meshgrid(xs, ys, indexing="ij"); return torch.stack([gx.reshape(-1), gy.reshape(-1)], -1)

## The clip, and the detector on frame 0

A fixed-camera pedestrian clip. We detect people on the first frame — those boxes are the seeds we'll carry.

In [ ]:
frames, fps = load_clip(VTEST, max_frames=48)
gray = to_gray(frames)
print("clip", tuple(frames.shape), f"@ {fps:.0f} fps")
show_frame(draw(frames[0], detect_people(frames[0])))

## Your task — `update_box`

Each box carries a grid of points; `lk_track` tells you where they went (`p0 → p1`, plus a `valid` mask).
Turn that into the box's new position. This is open-ended — at minimum **translate** the box by the points'
motion; you may also **rescale** it as the object nears/recedes. Be **robust**: some tracked points are
garbage, so a plain mean is a bad idea. Return `ok=False` when too few points survive to trust the update.

In [ ]:
def update_box(box, p0, p1, valid):
    """box [x1,y1,x2,y2] ; p0,p1 [N,2] point positions before/after ; valid [N] bool.
       Translate the box by the points' motion (robustly -- some tracks are bad); you may also
       rescale it. Return (new_box [4], ok: bool); ok=False if too few points survived."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# sanity check: move a box's points by a known shift and recover the translated box
box = torch.tensor([100., 100., 180., 260.]); pts = seed_points(box)
shift = torch.tensor([12., -5.])
new, ok = update_box(box, pts, pts + shift, torch.ones(len(pts), dtype=torch.bool))
assert ok, "update_box returned ok=False on a clean input"
assert torch.allclose(new, box + torch.tensor([12., -5., 12., -5.]), atol=1.0), new
print("update_box sanity check passed \u2713")

## The tracking loop (provided)

Detect on frame 0, carry every box with LK each frame, and **re-detect every `K` frames** to re-anchor.
`K` is the re-adapt frequency you'll study. Returns the boxes per frame and how many detector calls it cost.

In [ ]:
def track(frames, gray, K):
    boxes = detect_people(frames[0]); pts = [seed_points(b) for b in boxes]
    per_frame, n_det = [boxes], 1
    for i in range(1, frames.shape[0]):
        if i % K == 0:                                          # re-anchor with a fresh detection
            boxes = detect_people(frames[i]); pts = [seed_points(b) for b in boxes]; n_det += 1
        else:                                                   # carry boxes with optical flow
            nb, npts = [], []
            for b, p in zip(boxes, pts):
                p1, v = lk_track(gray[i-1], gray[i], p)
                b2, ok = update_box(b, p, p1, v)
                if ok: nb.append(b2); npts.append(p1)
            boxes = torch.stack(nb) if nb else torch.zeros(0, 4); pts = npts
        per_frame.append(boxes)
    return per_frame, n_det

## Baseline — detect once, then track with flow

Detect on frame 0, then carry those boxes with optical flow alone (`K = len(frames)`, so the detector never
runs again). Watch how much the cheap flow handles on its own: the people who were there at the start stay
tracked across the clip. The one thing it *cannot* do is notice anyone who walks in afterward — for that the
detector has to run again.

In [ ]:
per_frame, n_det = track(frames, gray, K=len(frames))           # never re-detect
print(f"{n_det} detector call(s)")
vis = torch.stack([draw(frames[i], per_frame[i], "red") for i in range(len(frames))])
show(vis, fps)

## The detector's price, and the speed/quality tradeoff in `K`

Re-running the detector is what lets the tracker pick up newcomers — but it is the expensive part. The extreme
is detecting on *every* frame (`K = 1`): always current, but how fast does it actually run? Time each setting
and compare them **relative to each other** (absolute fps depends on your hardware — CPU vs GPU). Frequent detection notices newcomers immediately but runs slowly; sparse
detection is fast but slow to react. Then watch the tracker at a sensible `K`.

In [ ]:
import time
print(f"the video was shot at {fps:.0f} fps  (absolute fps is hardware-dependent: CPU vs GPU)\n")
for K in [...]:           # YOUR CODE HERE: include K=1 (detect every frame) and a couple of larger values
    t = time.time(); per_frame, n_det = track(frames, gray, K); sec = time.time() - t
    print(f"K={K:>3}: {n_det:>2} detector calls, {len(frames)/sec:5.1f} fps  ({len(frames)/sec/fps:.2f}x clip rate)")

K = ...                   # YOUR CODE HERE: pick a K and watch the tracker
per_frame, n_det = track(frames, gray, K)
vis = torch.stack([draw(frames[i], per_frame[i], "lime") for i in range(len(frames))])
show(vis, fps)

---
**Submit** this notebook run top to bottom, with the video outputs intact.